# Z3 (C# / .NET) — Quantificateurs et preuves par refutation

**Twin C# de `Z3-Python-05-Quantifiers-Proofs.ipynb`** (parite .NET, marathon #4956, suite de `Z3-Python-04-Strings-Regex-Csharp`).

Ce notebook aborde les **quantificateurs** (`ForAll`, `Exists`) et la notion de **preuve formelle par refutation** avec le **moteur reel [Microsoft.Z3](https://github.com/Z3Prover/z3)** (NuGet, Z3 4.12.2.0). Une formule est *valide* (un theoreme) si sa **negation** est insatisfiable : on demande au solveur de trouver un contre-exemple, et s'il n'en existe aucun, la formule est demontree.

## Plan

1. Preuve de l'identite additive par refutation (`ForAll x, x+0 == x`).
2. Trois proprietes elementaires (multiplicatif, commutativite, neutre a droite).
3. `Exists` sat : existe-t-il un `x` tel que `x*x == 4` ? Piege de la variable liee.
4. `Exists` unsat : un carre reel est toujours positif (`x*x < 0` impossible).
5. Trichotomie et monotonie du carre sur les reels positifs.
6. Quantificateurs imbriques (`ForAll x, Exists y, y > x` = pas de plus grand reel).
7. Le cas honnete `unknown` : dernier theoreme de Fermat (`a^3 + b^3 == c^3`).
8. Model checking : skolemisation d'un `Exists x, ForAll y`.
9. Trois exercices.

> **API .NET** : `ctx.MkForall(new Expr[]{x}, body)` / `ctx.MkExists(new Expr[]{x}, body)` ou `x` est une constante creee via `ctx.MkConst("x", sort)` (le *body* utilise `x` directement, pas de de-Bruijn). `ctx.MkImplies`, `MkOr`, `MkAnd`, `MkNot` pour les connecteurs. Le timeout se regle via `s.Set("timeout", 3000)` ; la raison d'un `UNKNOWN` se lit via la propriete `s.ReasonUnknown`. Les constantes reelles/entieres retournees par `MkConst` sont statiquement `Expr` → a caster en `(ArithExpr)` pour `MkAdd`/`MkMul`/`MkLt`/etc.


In [1]:
#r "nuget: Microsoft.Z3"
using Microsoft.Z3;
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


## 1. Preuve par refutation : `ForAll x, x + 0 == x`

Pour prouver qu'une formule universelle est valide, on ajoute sa **negation** au solveur. Si le solveur repond `UNSATISFIABLE`, c'est qu'il n'existe aucun contre-exemple : la formule est donc un theoreme. Ici `x + 0 == x` pour tout reel `x`.


In [2]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);

// La formule que nous voulons prouver : pour tout x reel, x + 0 == x
var body = ctx.MkEq(ctx.MkAdd(x, ctx.MkReal(0)), x);
var formule = ctx.MkForall(new Expr[]{ x }, body);
Console.WriteLine("Formule a prouver : " + formule);

// Technique : verifier que la NEGATION est insatisfiable
var s = ctx.MkSolver();
s.Add(ctx.MkNot(formule));  // Existe-t-il un x tel que x + 0 != x ?

Status resultat = s.Check();
Console.WriteLine("Negation de la formule : " + resultat);

if (resultat == Status.UNSATISFIABLE)
    Console.WriteLine("=> Aucun contre-exemple trouve : la formule est VALIDE (theoreme).");
else if (resultat == Status.SATISFIABLE)
    Console.WriteLine("=> Contre-exemple trouve : " + s.Model);
else
    Console.WriteLine("=> Z3 ne peut pas conclure (unknown).");


Formule a prouver : (forall ((x Real)) (= (+ x 0.0) x))


Negation de la formule : UNSATISFIABLE


=> Aucun contre-exemple trouve : la formule est VALIDE (theoreme).


## 2. Trois proprietes elementaires

On automatise la preuve par refutation sur trois proprietes : l'identite multiplicative (`x * 1 == x`), la commutativite de l'addition (`x + y == y + x`), et le neutre additif a droite (`0 + x == x`). Pour chacune, on nie la formule universelle et on regarde le verdict du solveur.


In [3]:
using Microsoft.Z3;
using System.Collections.Generic;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);
var y = (ArithExpr)ctx.MkConst("y", ctx.RealSort);

var proprietes = new (string nom, Quantifier formule)[]{
    ("Identite multiplicative : x * 1 == x",
        ctx.MkForall(new Expr[]{ x }, ctx.MkEq(ctx.MkMul(x, ctx.MkReal(1)), x))),
    ("Commutativite de l addition : x + y == y + x",
        ctx.MkForall(new Expr[]{ x, y }, ctx.MkEq(ctx.MkAdd(x, y), ctx.MkAdd(y, x)))),
    ("Neutre additif a droite : 0 + x == x",
        ctx.MkForall(new Expr[]{ x }, ctx.MkEq(ctx.MkAdd(ctx.MkReal(0), x), x))),
};

foreach (var (nom, formule) in proprietes)
{
    var s = ctx.MkSolver();
    s.Add(ctx.MkNot(formule));
    var res = s.Check();
    string statut = res == Status.UNSATISFIABLE ? "VALIDE" : (res == Status.SATISFIABLE ? "FAUSSE" : "UNKNOWN");
    Console.WriteLine(nom);
    Console.WriteLine("  => " + statut + " (negation = " + res + ")");
    Console.WriteLine();
}


Identite multiplicative : x * 1 == x


  => VALIDE (negation = UNSATISFIABLE)


Commutativite de l addition : x + y == y + x


  => VALIDE (negation = UNSATISFIABLE)


Neutre additif a droite : 0 + x == x


  => VALIDE (negation = UNSATISFIABLE)


## 3. `Exists` satisfiable : existe-t-il `x` tel que `x*x == 4` ?

Le quantificateur existentiel `MkExists` demontre l'existence d'un temoin. **Piege classique** : la variable liee par `Exists` n'apparait **pas** dans le modele (elle est quantifiee) — `m.Evaluate(x)` ne donne pas de temoin lisible. Pour obtenir une valeur concrete, on **skolemise** : on reasserte la meme contrainte sur une constante *libre*, dont la valeur est alors lisible dans le modele.


In [4]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);

// Etape 1 : prouver l existence avec le quantificateur existentiel
var formule = ctx.MkExists(new Expr[]{ x }, ctx.MkEq(ctx.MkMul(x, x), ctx.MkReal(4)));
var s = ctx.MkSolver();
s.Add(formule);
var res = s.Check();
Console.WriteLine("Formule   : " + formule);
Console.WriteLine("Existence : " + res + " (un temoin existe)");

// Piege : x est LIEE par Exists -> pas dans le modele
Console.WriteLine("m[x] (x lie par Exists) : " + s.Model.Evaluate(x) + "  <- aucun temoin lisible ici");

// Etape 2 : skolemisation sur une constante LIBRE
var racine = (ArithExpr)ctx.MkConst("racine", ctx.RealSort);
var s2 = ctx.MkSolver();
s2.Add(ctx.MkEq(ctx.MkMul(racine, racine), ctx.MkReal(4)));
s2.Check();
var temoin = s2.Model.Evaluate(racine);
Console.WriteLine("Temoin concret (constante libre) : racine = " + temoin);
Console.WriteLine("Verification : " + temoin + " * " + temoin + " = " + ctx.MkMul((ArithExpr)temoin, (ArithExpr)temoin).Simplify() + " (attendu : 4)");


Formule   : (exists ((x Real)) (= (* x x) 4.0))


Existence : SATISFIABLE (un temoin existe)


m[x] (x lie par Exists) : x  <- aucun temoin lisible ici


Temoin concret (constante libre) : racine = 2


Verification : 2 * 2 = 4 (attendu : 4)


## 4. `Exists` insatisfiable : un carre negatif n'existe pas

A l'inverse, `MkExists(new Expr[]{ x }, x*x < 0)` est insatisfiable sur les reels : un carre est toujours positif ou nul. Le solveur repond `UNSATISFIABLE`, demontrant que la formule existentielle est fausse.


In [5]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);

var formule = ctx.MkExists(new Expr[]{ x }, ctx.MkLt(ctx.MkMul(x, x), ctx.MkReal(0)));
Console.WriteLine("Formule : " + formule);
Console.WriteLine("(Un carre reel est toujours positif ou nul)");

var s = ctx.MkSolver();
s.Add(formule);
var res = s.Check();
Console.WriteLine("Resultat : " + res);
if (res == Status.UNSATISFIABLE)
    Console.WriteLine("=> Aucun reel x tel que x*x < 0 : la formule est FAUSSE.");
else if (res == Status.SATISFIABLE)
    Console.WriteLine("=> Temoin trouve : " + s.Model);


Formule : (exists ((x Real)) (< (* x x) 0.0))


(Un carre reel est toujours positif ou nul)


Resultat : UNSATISFIABLE


=> Aucun reel x tel que x*x < 0 : la formule est FAUSSE.


## 5. Trichotomie et monotonie du carre

Deux theoremes classiques prouves par refutation. **Trichotomie** : pour tout reel `x`, exactement un des trois cas `x < 0`, `x == 0`, `x > 0` est vrai. **Monotonie du carre** : pour tous `x, y >= 0`, si `x <= y` alors `x*x <= y*y`. On nie chaque formule universelle et on constate `UNSATISFIABLE`.


In [6]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);
var y = (ArithExpr)ctx.MkConst("y", ctx.RealSort);

// Preuve 1 : trichotomie sur les reels
var trichotomie = ctx.MkForall(new Expr[]{ x }, ctx.MkOr(
    ctx.MkLt(x, ctx.MkReal(0)),
    ctx.MkEq(x, ctx.MkReal(0)),
    ctx.MkGt(x, ctx.MkReal(0))));
Console.WriteLine("Theoreme de trichotomie : " + trichotomie);
var s = ctx.MkSolver();
s.Add(ctx.MkNot(trichotomie));
var res = s.Check();
Console.WriteLine("Negation = " + res);
Console.WriteLine("=> Trichotomie : " + (res == Status.UNSATISFIABLE ? "VALIDE" : "NON PROUVEE"));
Console.WriteLine();

// Preuve 2 : monotonie du carre sur les reels positifs
var monotonicite = ctx.MkForall(new Expr[]{ x, y }, ctx.MkImplies(
    ctx.MkAnd(ctx.MkGe(x, ctx.MkReal(0)), ctx.MkGe(y, ctx.MkReal(0)), ctx.MkLe(x, y)),
    ctx.MkLe(ctx.MkMul(x, x), ctx.MkMul(y, y))));
Console.WriteLine("Monotonicite du carre (x >= 0) : " + monotonicite);
var s2 = ctx.MkSolver();
s2.Add(ctx.MkNot(monotonicite));
var res2 = s2.Check();
Console.WriteLine("Negation = " + res2);
Console.WriteLine("=> Monotonicite : " + (res2 == Status.UNSATISFIABLE ? "VALIDE" : "NON PROUVEE"));


Theoreme de trichotomie : (forall ((x Real)) (or (< x 0.0) (= x 0.0) (> x 0.0)))


Negation = UNSATISFIABLE


=> Trichotomie : VALIDE


Monotonicite du carre (x >= 0) : (forall ((x Real) (y Real))
  (=> (and (>= x 0.0) (>= y 0.0) (<= x y)) (<= (* x x) (* y y))))


Negation = UNSATISFIABLE


=> Monotonicite : VALIDE


## 6. Quantificateurs imbriques : pas de plus grand reel

Une formule peut imbriquer `ForAll` et `Exists`. L'enonce « il n'existe pas de plus grand reel » se formalise : `ForAll x, Exists y, y > x` — pour tout `x`, on peut toujours trouver un `y` strictement plus grand. On nie la formule entiere et le solveur repond `UNSATISFIABLE`.


In [7]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);
var y = (ArithExpr)ctx.MkConst("y", ctx.RealSort);

// ForAll x, Exists y, y > x
var pasDePlusGrand = ctx.MkForall(new Expr[]{ x },
    ctx.MkExists(new Expr[]{ y }, ctx.MkGt(y, x)));
Console.WriteLine("Formule : " + pasDePlusGrand);

var s = ctx.MkSolver();
s.Add(ctx.MkNot(pasDePlusGrand));
var res = s.Check();
Console.WriteLine("Negation = " + res);
if (res == Status.UNSATISFIABLE)
    Console.WriteLine("=> VALIDE : il n existe pas de plus grand reel (theoreme).");
else if (res == Status.SATISFIABLE)
    Console.WriteLine("=> FAUSSE : contre-exemple trouve. " + s.Model);
else
    Console.WriteLine("=> Z3 ne peut pas conclure (unknown).");


Formule : (forall ((x Real)) (exists ((y Real)) (> y x)))


Negation = UNSATISFIABLE


=> VALIDE : il n existe pas de plus grand reel (theoreme).


## 7. Le cas honnete `unknown` : Fermat

L'arithmetique non lineaire entiere est **indecidable en general** : Z3 ne peut pas toujours trancher. Existe-t-il des entiers `a, b, c > 1` tels que `a^3 + b^3 == c^3` ? Le dernier theoreme de Fermat dit non, mais cette preuve depasse Z3. Sans budget de temps, Z3 chercherait indefiniment ; on borne donc avec un timeout, et le solveur repond honnetement `UNKNOWN` avec une raison (`timeout`). Ce n'est **pas un bug** mais une limite fondamentale de la decision automatique.


In [8]:
using Microsoft.Z3;
var ctx = new Context();
var a = (ArithExpr)ctx.MkConst("a", ctx.IntSort);
var b = (ArithExpr)ctx.MkConst("b", ctx.IntSort);
var c = (ArithExpr)ctx.MkConst("c", ctx.IntSort);

var s = ctx.MkSolver();
s.Set("timeout", 3000);  // 3 secondes ; au-dela, Z3 abandonne
s.Add(ctx.MkGt(a, ctx.MkInt(1)), ctx.MkGt(b, ctx.MkInt(1)), ctx.MkGt(c, ctx.MkInt(1)));
var a3 = ctx.MkMul(a, ctx.MkMul(a, a));
var b3 = ctx.MkMul(b, ctx.MkMul(b, b));
var c3 = ctx.MkMul(c, ctx.MkMul(c, c));
s.Add(ctx.MkEq(ctx.MkAdd(a3, b3), c3));

Console.WriteLine("Probleme : a^3 + b^3 == c^3  avec a, b, c > 1");
var res = s.Check();
Console.WriteLine("Resultat : " + res);

if (res == Status.SATISFIABLE)
    Console.WriteLine("=> Temoin trouve : " + s.Model);
else if (res == Status.UNSATISFIABLE)
    Console.WriteLine("=> Aucune solution (prouve).");
else
{
    Console.WriteLine("=> UNKNOWN : Z3 abandonne (raison : " + s.ReasonUnknown + ").");
    Console.WriteLine("   L arithmetique non lineaire entiere est indecidable en general :");
    Console.WriteLine("   ce n est PAS un bug, mais une limite fondamentale de la decision automatique.");
}


Probleme : a^3 + b^3 == c^3  avec a, b, c > 1


Resultat : UNKNOWN


=> UNKNOWN : Z3 abandonne (raison : timeout).


   L arithmetique non lineaire entiere est indecidable en general :


   ce n est PAS un bug, mais une limite fondamentale de la decision automatique.


## 8. Model checking : skolemiser un `Exists x, ForAll y`

On cherche s'il existe un `x` tel que pour tout `y >= 0`, `x <= y`. Le `Exists x` externe est le temoin recherche. Pour lire sa valeur, on le **skolemise** : on declare `x` comme constante *libre* et le `ForAll` ne porte plus que sur `y`. Le solveur trouve alors un modele (par exemple `x = 0`), dont la valeur devient lisible via `m.Evaluate(x)`.


In [9]:
using Microsoft.Z3;
var ctx = new Context();
var x = (ArithExpr)ctx.MkConst("x", ctx.RealSort);
var y = (ArithExpr)ctx.MkConst("y", ctx.RealSort);

// ForAll y >= 0 => x <= y  (x skolemise, libre)
var contrainte = ctx.MkForall(new Expr[]{ y },
    ctx.MkImplies(ctx.MkGe(y, ctx.MkReal(0)), ctx.MkLe(x, y)));
Console.WriteLine("Contrainte sur x (temoin libre) : " + contrainte);

var s = ctx.MkSolver();
s.Add(contrainte);
var res = s.Check();
Console.WriteLine("Resultat : " + res);

if (res == Status.SATISFIABLE)
{
    var m = s.Model;
    var valX = m.Evaluate(x);  // x est libre -> sa valeur est lisible
    Console.WriteLine("Modele trouve : x = " + valX);
    Console.WriteLine("Interpretation : " + valX + " est <= a tout y >= 0 (un minorant des reels positifs)");
    // Verifier x <= 1/10 et x <= 1/1000 contre le modele
    Console.WriteLine("Verification : x <= 1/10 ? " + m.Evaluate(ctx.MkLe(x, ctx.MkReal(1, 10))));
    Console.WriteLine("Verification : x <= 1/1000 ? " + m.Evaluate(ctx.MkLe(x, ctx.MkReal(1, 1000))));
}
else if (res == Status.UNSATISFIABLE)
    Console.WriteLine("=> Aucun modele : la formule est fausse.");
else
    Console.WriteLine("=> Z3 ne peut pas conclure (unknown).");


Contrainte sur x (temoin libre) : (forall ((y Real)) (=> (>= y 0.0) (<= x y)))


Resultat : SATISFIABLE


Modele trouve : x = 0


Interpretation : 0 est <= a tout y >= 0 (un minorant des reels positifs)


Verification : x <= 1/10 ? true


Verification : x <= 1/1000 ? true


## Exercices

Trois exercices a completer. Les stubs retournent `null`.


In [10]:
// EXERCICE 1 : Prouver l identite multiplicative par refutation.
// Verifier que ForAll(x, x * 1 == x) est valide sur les reels.
// Indice : niez la formule et regardez le verdict.
// Etape 1 : declarer x = (ArithExpr)ctx.MkConst("x", ctx.RealSort)
// Etape 2 : formule = ctx.MkForall(new Expr[]{x}, ctx.MkEq(ctx.MkMul(x, ctx.MkReal(1)), x))
// Etape 3 : s.Add(ctx.MkNot(formule)), return s.Check() == Status.UNSATISFIABLE
bool? ProuverIdentiteMultiplicative(Context ctx)
{
    // TODO etudiant : implementez la preuve par refutation
    return null;  // TODO etudiant : remplacer par true (valide) ou false (fause)
}

var r1 = ProuverIdentiteMultiplicative(new Context());
Console.WriteLine("Exercice 1 (x * 1 == x valide ?) : " + (r1.HasValue ? r1.Value.ToString() : "(a completer)"));


Exercice 1 (x * 1 == x valide ?) : (a completer)


In [11]:
// EXERCICE 2 : Verifier si un carre negatif existe sur les reels.
// Existe-t-il x reel tel que x*x < 0 ? Retourne true si sat, false sinon.
// Indice : MkExists + MkLt(x*x, 0) + Check.
// Etape 1 : declarer x = (ArithExpr)ctx.MkConst("x", ctx.RealSort)
// Etape 2 : s.Add(ctx.MkExists(new Expr[]{x}, ctx.MkLt(ctx.MkMul(x,x), ctx.MkReal(0))))
// Etape 3 : return s.Check() == Status.SATISFIABLE
bool? ExisteCarreNegatif(Context ctx)
{
    // TODO etudiant : implementez la verification
    return null;  // TODO etudiant : remplacer par true ou false
}

var r2 = ExisteCarreNegatif(new Context());
Console.WriteLine("Exercice 2 (un carre negatif existe ?) : " + (r2.HasValue ? r2.Value.ToString() : "(a completer)"));


Exercice 2 (un carre negatif existe ?) : (a completer)


In [12]:
// EXERCICE 3 : Prouver qu il n existe pas de plus grand reel.
// Verifier que ForAll(x, Exists(y, y > x)) est valide sur les reels.
// Indice : niez toute la formule et regardez le verdict.
// Etape 1 : declarer x, y = (ArithExpr)ctx.MkConst(..., ctx.RealSort)
// Etape 2 : nest = ctx.MkForall(new Expr[]{x}, ctx.MkExists(new Expr[]{y}, ctx.MkGt(y, x)))
// Etape 3 : s.Add(ctx.MkNot(nest)), return s.Check() == Status.UNSATISFIABLE
bool? ProuverPasDePlusGrandReel(Context ctx)
{
    // TODO etudiant : implementez la preuve par refutation
    return null;  // TODO etudiant : remplacer par true ou false
}

var r3 = ProuverPasDePlusGrandReel(new Context());
Console.WriteLine("Exercice 3 (pas de plus grand reel, valide ?) : " + (r3.HasValue ? r3.Value.ToString() : "(a completer)"));


Exercice 3 (pas de plus grand reel, valide ?) : (a completer)


## Conclusion

Ce twin C# couvre les **quantificateurs** (`MkForall`/`MkExists` sur `new Expr[]{ constantes }`) et la **preuve par refutation** (une formule universelle est valide ssi sa negation est insatisfiable) avec le moteur reel Microsoft.Z3. On a prouve l'identite additive/multiplicative, la commutativite, la trichotomie, la monotonie du carre, et l'absence de plus grand reel ; traite le piege de la variable liee par `Exists` (skolemisation) ; et affronte honnetement la limite `UNKNOWN` de l'arithmetique non lineaire entiere (Fermat, avec `ReasonUnknown`).

**Complementarite** : le twin Python utilise `ForAll([x], ...)` / `Exists([x], ...)` / `is_true(simplify(...))` (API pythonique, `unsat`/`sat`/`unknown` litteraux) ; ce twin C# montre l'API .NET (`MkForall(new Expr[]{x}, body)` / `Status.UNSATISFIABLE` enum / `ReasonUnknown` propriete / `Set("timeout", N)`), avec le casting `(ArithExpr)` systematique des constantes reelles/entieres. Les deux executent le **meme** moteur Z3 et demontrent les memes theoremes.
